In [101]:
import random
import numpy as np
import math

In [102]:
import numpy as np

def init_params_es(n):
    # Population size (lambda)
    pop_size = 4 + int(3 * np.log(n))

    # Number of parents (mu)
    mu = pop_size // 2

    # w_i: weights for mean update
    weights = np.log(mu + 0.5) - np.log(np.arange(1, mu + 1))
    weights /= np.sum(weights) # Normalize to sum 1

    mu_w = 1 / np.sum(weights ** 2)
    c_1 = 2 / n ** 2
    c_mu = mu_w / n ** 2

    # Learning rates
    c_sigma = (mu_w + 2) / (n + mu_w + 5)
    c_c = (4 + mu_w / n) / (n + 4 + 2 * mu_w / n)

    return {
        "n": n,
        "pop_size": pop_size, "mu": mu, "weights": weights, "mu_w": mu_w,
        "c_1": c_1, "c_mu": c_mu, "c_sigma": c_sigma, "c_c": c_c,
        "m": np.random.uniform(-5, 5, n),       # Init core
        "sigma": 0.5,                           # Init step size
        "C": np.eye(n),                         # Covariance matrix
        "p_c": np.zeros(n),                     # Evol path for C
        "p_sigma": np.zeros(n)                  # Evol path for sigma
    }

In [103]:
def target_function(x):
    return sum(100.0 * (x[1:] - x[:-1]**2.0)**2.0 + (1.0 - x[:-1])**2.0)

def evolutionary_strategy(
        n_vars,
        max_gens = 300,
):
    # Init params
    params = init_params_es(n=n_vars)

    for gen in range(max_gens):

        # Step 2
        vals, vecs = np.linalg.eigh(params["C"])
        D = np.diag(np.sqrt(np.maximum(0, vals)))
        BD = np.dot(vecs, D)

        z = np.random.randn(params["pop_size"], n_vars)
        y = np.dot(z, BD.T)
        x = params["m"] + params["sigma"] * y

        # Step 3: Calculate fitness
        fitness = np.array([target_function(ind) for ind in x])

        # Step 4: Selection
        indices = np.argsort(fitness)
        y_sel = y[indices[:params["mu"]]]
        x_sel = x[indices[:params["mu"]]]

        # Step 5: Update core m
        y_w = np.sum(y_sel * params["weights"][:, np.newaxis], axis=0)
        # params["m"] = params["m"] + params["sigma"] * y_w
        params["m"] = np.sum(x_sel * params["weights"][:, np.newaxis], axis=0)

        # Step 6: Update p_sigma
        C_inv_half = np.linalg.multi_dot([vecs, np.diag(1.0 / np.sqrt(np.maximum(1e-10, vals))), vecs.T])
        params["p_sigma"] = (1 - params["c_sigma"]) * params["p_sigma"] + \
            np.sqrt(1 - (1 - params["c_sigma"]) ** 2 ) * np.sqrt(params["mu_w"]) * np.dot(C_inv_half, y_w)

        # Step 7: Update sigma
        p_sigma_norm = np.linalg.norm(params["p_sigma"])
        params["sigma"] = params["sigma"] * np.exp(params["c_sigma"] * (p_sigma_norm  / np.sqrt(params["n"]) - 1))

        # Step 8: Update p_c
        params["p_c"] = (1 - params["c_c"]) * params["p_c"] + \
            np.sqrt(1 - (1 - params["c_c"]) ** 2 ) * np.sqrt(params["mu_w"]) * y_w

        # Step 9: Update C
        rank_1 = params["c_1"] * np.outer(params["p_c"], params["p_c"])
        rank_mu = np.zeros((n_vars, n_vars))
        for i in range(params["mu"]):
            rank_mu += params["weights"][i] * np.outer(y_sel[i], y_sel[i])

        params["C"] = (1 - params["c_1"] - params["c_mu"]) * params["C"] + rank_1 + params["c_mu"]  *rank_mu

        if gen % 30 == 0 or gen == max_gens - 1:
            best_f = np.min(fitness)
            print(f"Gen {gen} - Best fitness: {best_f:.4f} | Sigma: {params['sigma']:.4f} | Mean: {params['m']}")

    return params['m'], np.min(fitness)

In [104]:
best_m, best_f = evolutionary_strategy(n_vars=10, max_gens=10000)
print(f"Best mean: {best_m} | Best fitness: {best_f}")

Gen 0 - Best fitness: 1691808.6755 | Sigma: 0.4799 | Mean: [ 3.52092566  2.57308845 -3.60135624 -4.28817041 -0.65393025  4.20190039
 -3.68214264 -0.29411753 -5.1562031   4.20659201 -3.77475988 -5.07040477
 -4.42289233  4.69280406  3.42698891 -3.08724505  1.98663852  4.87489037
  4.47286897  0.25227526  2.23206065 -3.77767416  4.36122829 -3.44258088
  0.84660548  4.30178266  3.14920421 -0.35733644 -2.20902313 -3.74342056
  3.86353489 -3.36409703  0.90047494 -3.56004835 -3.64450346  3.90541917
  4.26544863 -1.12764111 -4.36253914 -3.45652688  1.98585734  1.63091835
 -4.37116333 -4.47540369 -0.43478539 -2.30569335  3.10988572  0.62700553
 -2.41748623  4.51088654  3.80637218 -2.63345043 -4.38713176  2.91478776
 -2.49127115  3.02228315  4.3200363  -1.53198029  2.67235516 -3.16945099
 -1.49218208  1.74175673 -0.73033381 -4.10688962 -1.45963061  1.02916986
 -3.54413142  4.34540712 -1.74008908 -0.8067008  -2.96049153  2.55110953
 -4.63625896 -3.46275852  3.8550658   4.30663112  4.64041561  5.0